# 📊 Clase 3 — Detección y Tratamiento de Outliers (Python)

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · drdarioezequieldiaz@gmail.com

**Fecha:** Jueves 23 de abril de 2026

---

### Objetivos

Este notebook despliega el aparato completo de técnicas para detección y tratamiento de valores atípicos desarrollado en el apunte teórico. Persigue cinco metas articuladas:

1. **Caracterizar formalmente** la naturaleza de los outliers en datos financieros mediante análisis de curtosis, tests de normalidad y evaluación empírica de colas pesadas.

2. **Aplicar sistemáticamente** los métodos univariados (IQR, IQR contextualizado, z-score clásico, z-score MAD, criterio de Chauvenet, test de Grubbs), exhibiendo empíricamente el fenómeno de *masking* y el concepto de *breakdown point*.

3. **Desplegar los métodos multivariados** (Mahalanobis clásica vs robusta MCD, LOF, Isolation Forest, DBSCAN, One-Class SVM) comparando fortalezas relativas y construyendo un esquema de consenso.

4. **Ilustrar la Ley de Benford** como herramienta forense para detección de manipulación en datos contables.

5. **Cuantificar el impacto** de los outliers sobre estimadores clásicos (media, varianza, correlación, OLS) y contrastar las **estrategias de tratamiento** (trimming, winsorización, transformaciones Box-Cox / Yeo-Johnson, regresión robusta de Huber) en términos de sesgo, robustez y capacidad predictiva sobre scoring crediticio.

> **Referencia teórica:** Apunte Clase 3 — *Detección y Tratamiento de Valores Atípicos en Datos Financieros* (Dr. Darío E. Díaz, 2026).

### Estructura del cuaderno

| Parte | Contenido |
|-------|-----------|
| **I**   | Fundamentos: colas pesadas, normalidad, falla del 3-sigma |
| **II**  | Métodos univariados: IQR, z-score MAD, Chauvenet, Grubbs |
| **III** | Métodos multivariados: Mahalanobis clásica y MCD, LOF, IForest, DBSCAN, OC-SVM |
| **IV**  | Método específico financiero: Ley de Benford |
| **V**   | Impacto estadístico: sesgo, leverage, distancia de Cook |
| **VI**  | Estrategias de tratamiento: trimming, winsorización, transformaciones, robustez |
| **VII** | Evaluación en scoring crediticio y síntesis |

---
## 1. Configuración del entorno

Se importan las bibliotecas necesarias: `scikit-learn` provee los algoritmos de detección multivariada (`EllipticEnvelope`, `IsolationForest`, `LocalOutlierFactor`, `OneClassSVM`, `DBSCAN`); `scipy.stats` aporta los tests formales (Grubbs, Jarque-Bera, Shapiro); `statsmodels` suministra la regresión robusta de Huber. Adicionalmente se fija una paleta cromática coherente con el apunte.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. CONFIGURACIÓN DEL ENTORNO
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import seaborn as sns

# Detección de outliers — scikit-learn
from sklearn.datasets import fetch_openml
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope, MinCovDet, EmpiricalCovariance
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, LabelEncoder, PowerTransformer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

# Tests estadísticos
from scipy import stats
from scipy.stats import (median_abs_deviation, shapiro, jarque_bera,
                         anderson, norm, t, chi2, boxcox, yeojohnson)
from scipy.stats.mstats import winsorize

# Modelos robustos
import statsmodels.api as sm
from statsmodels.robust.robust_linear_model import RLM
from statsmodels.robust.norms import HuberT, TukeyBiweight

import warnings
warnings.filterwarnings('ignore')

# ───────────── Estilo visual ─────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titleweight'] = 'bold'

# Paleta Midnight Executive del apunte
COL_NAVY    = '#1A2744'
COL_TEAL    = '#0C7C8A'
COL_TEAL_L  = '#14A3B8'
COL_AMBER   = '#F59E0B'
COL_RED     = '#E76F51'
COL_PURPLE  = '#7C3AED'
COL_GREEN   = '#2a9d8f'
COL_GRAY    = '#6B7280'
COL_GOOD, COL_BAD, COL_ACC = COL_GREEN, COL_RED, COL_AMBER

RS = 42  # random seed

print("✅ Entorno configurado")
print(f"   scikit-learn, scipy, statsmodels cargados")
print(f"   Paleta visual Midnight Executive activa")
print(f"   Semilla aleatoria fija: {RS}")

---
## 2. Carga del dataset German Credit

Conservamos el German Credit Dataset como caso de estudio principal, respetando el principio pedagógico de trabajar siempre desde un dataset completo (sin valores faltantes) para poder introducir artificialmente escenarios controlados. Además, construimos un **dataset auxiliar de transacciones simuladas** que servirá para ejemplificar la Ley de Benford y los outliers colectivos.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. CARGA Y EXPLORACIÓN INICIAL
# ══════════════════════════════════════════════════════════════
# Estrategia de carga con múltiples fallbacks:
#   (a) sklearn.fetch_openml desde OpenML (requiere internet)
#   (b) CSV local 'german_credit.csv' si existe en el directorio
#   (c) Generación sintética con distribuciones calibradas (último recurso)

def cargar_german_credit():
    # Opción (a): OpenML
    try:
        credit = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
        df_ = credit.frame.copy()
        print("✓ Dataset cargado desde OpenML")
        return df_
    except Exception as e:
        print(f"  (OpenML no disponible: {type(e).__name__})")
    # Opción (b): CSV local
    import os
    for path in ['german_credit.csv', 'german_credit_synthetic.csv',
                 '/content/german_credit.csv']:
        if os.path.exists(path):
            df_ = pd.read_csv(path)
            print(f"✓ Dataset cargado desde CSV local: {path}")
            return df_
    # Opción (c): generación sintética
    print("⚠ Generando dataset sintético con distribuciones calibradas del German Credit...")
    np.random.seed(42)
    n = 1000
    df_ = pd.DataFrame({
        'duration':               np.clip(np.random.lognormal(2.9, 0.5, n).round(), 4, 72).astype(int),
        'credit_amount':          np.clip(np.random.lognormal(7.9, 0.7, n), 250, 20000).round().astype(int),
        'installment_commitment': np.random.choice([1, 2, 3, 4], n, p=[0.14, 0.16, 0.22, 0.48]),
        'residence_since':        np.random.choice([1, 2, 3, 4], n, p=[0.13, 0.32, 0.17, 0.38]),
        'age':                    np.clip(np.random.gamma(5, 7, n) + 19, 19, 75).round().astype(int),
        'existing_credits':       np.random.choice([1, 2, 3, 4], n, p=[0.63, 0.33, 0.03, 0.01]),
        'num_dependents':         np.random.choice([1, 2], n, p=[0.85, 0.15]),
        'class':                  np.random.choice(['good', 'bad'], n, p=[0.7, 0.3]),
    })
    df_['credit_amount'] = (df_['credit_amount'] * (0.6 + 0.02 * df_['duration'])).round().astype(int).clip(250, 25000)
    return df_


df = cargar_german_credit()

# Variables numéricas relevantes para el análisis de outliers
num_vars = ['duration', 'credit_amount', 'installment_commitment',
            'residence_since', 'age', 'existing_credits', 'num_dependents']

# Codificación: y = 1 para default (bad credit), y = 0 para cumplimiento
y = (df['class'] == 'bad').astype(int).values

print("=" * 70)
print("DATASET GERMAN CREDIT — Estadísticos univariados")
print("=" * 70)
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Tasa de default: {y.mean()*100:.1f}%\n")

stats_desc = df[num_vars].describe().T[['mean', 'std', '50%', 'min', 'max']]
stats_desc.columns = ['Media', 'Desvío', 'Mediana', 'Mín', 'Máx']
stats_desc['Asimetría'] = [df[v].skew() for v in num_vars]
stats_desc['Curtosis']  = [df[v].kurtosis() + 3 for v in num_vars]  # Pearson (normal = 3)
print(stats_desc.round(2))

### 📝 Primera lectura de los estadísticos

Varias variables exhiben **curtosis sustancialmente superior a 3** (valor de la distribución normal). La variable `credit_amount` presenta asimetría positiva marcada y curtosis que la aleja de la gaussianidad: indicio preliminar de distribución con cola derecha pesada, característica habitual en montos monetarios. Este hallazgo anticipa que la aplicación ingenua del criterio 3-sigma producirá excesos de falsos positivos, tal como demostraremos en la siguiente sección.

---
# PARTE I — Fundamentos empíricos

## 3. Análisis de colas pesadas: curtosis y tests de normalidad

Antes de aplicar cualquier método de detección debemos caracterizar la distribución subyacente de cada variable. La distinción entre "valor extremo legítimo de una cola pesada" y "outlier genuino" depende críticamente de esta caracterización.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. TESTS FORMALES DE NORMALIDAD
# ══════════════════════════════════════════════════════════════

print("=" * 85)
print("TESTS DE NORMALIDAD (H0: los datos provienen de una distribución normal)")
print("=" * 85)
print(f"{'Variable':<26s} {'Curtosis':>9s} {'Shapiro p':>12s} {'Jarque-Bera p':>15s} {'Normal?':>10s}")
print("-" * 85)

for v in num_vars:
    x = df[v].astype(float).values
    kurt = stats.kurtosis(x, fisher=False)           # Pearson (normal = 3)
    # Shapiro-Wilk (truncar a 5000 para evitar sobre-detección en muestras grandes)
    n_sample = min(len(x), 5000)
    sw_stat, sw_p = shapiro(x[:n_sample])
    # Jarque-Bera
    jb_stat, jb_p = jarque_bera(x)
    is_normal = "✓" if (sw_p > 0.05 and jb_p > 0.05) else "✗"
    print(f"  {v:<24s} {kurt:>9.2f} {sw_p:>12.2e} {jb_p:>15.2e} {is_normal:>10s}")

print("\nLectura: ninguna variable satisface la hipótesis de normalidad.")
print("Curtosis de referencia: distribución normal = 3; t-Student(df=4) ≈ 9.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.1 VISUALIZACIÓN DE LA NO-NORMALIDAD
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
vars_show = ['credit_amount', 'duration', 'age']

for j, v in enumerate(vars_show):
    x = df[v].astype(float).values
    # Histograma con curva normal superpuesta
    axes[0, j].hist(x, bins=40, density=True, color=COL_TEAL, alpha=0.6,
                    edgecolor='white', label='Empírica')
    xx = np.linspace(x.min(), x.max(), 200)
    axes[0, j].plot(xx, norm.pdf(xx, x.mean(), x.std()), color=COL_RED,
                    lw=2.2, label='Normal ajustada')
    axes[0, j].set_title(f'{v} — Histograma vs Normal')
    axes[0, j].set_xlabel(v); axes[0, j].set_ylabel('Densidad')
    axes[0, j].legend(fontsize=9)

    # Q-Q plot
    stats.probplot(x, dist="norm", plot=axes[1, j])
    axes[1, j].get_lines()[0].set_markerfacecolor(COL_TEAL)
    axes[1, j].get_lines()[0].set_markeredgecolor(COL_TEAL)
    axes[1, j].get_lines()[0].set_markersize(4)
    axes[1, j].get_lines()[1].set_color(COL_RED)
    axes[1, j].get_lines()[1].set_linewidth(2)
    axes[1, j].set_title(f'Q-Q plot — {v}')

plt.suptitle('Evaluación visual de normalidad: histograma + Q-Q plot',
             fontweight='bold', fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

### 📝 Interpretación

Los Q-Q plots revelan la estructura de las colas: las curvas que se desvían hacia arriba en el extremo derecho confirman la presencia de **cola derecha pesada** (leptocurtosis). La variable `credit_amount` exhibe la desviación más severa: la cola empírica yace muy por encima de la recta de referencia gaussiana en los cuantiles altos, significando que la distribución asigna una probabilidad sustancialmente mayor a los valores elevados que la que asignaría una normal con idéntica media y varianza.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.2 FALLA DEL CRITERIO 3-SIGMA BAJO COLAS PESADAS
# ══════════════════════════════════════════════════════════════

# Simulación: comparar proporción esperada |z|>3 bajo normal vs t-Student(df=4)
np.random.seed(RS)
n_sim = 100_000

samples = {
    'Normal N(0,1)':         np.random.normal(0, 1, n_sim),
    't-Student (df=4)':      np.random.standard_t(4, n_sim),
    't-Student (df=3)':      np.random.standard_t(3, n_sim),
    'Log-Normal(0,1)':       np.random.lognormal(0, 1, n_sim),
}

print("=" * 70)
print("FALLA DEL CRITERIO 3-SIGMA BAJO DISTRIBUCIONES DE COLAS PESADAS")
print("=" * 70)
print(f"{'Distribución':<22s} {'Curtosis':>10s} {'% |z|>3':>12s} {'% |z|>3.5':>12s}")
print("-" * 60)

for name, x in samples.items():
    z = (x - x.mean()) / x.std()
    kurt = stats.kurtosis(x, fisher=False)
    pct_3 = (np.abs(z) > 3).mean() * 100
    pct_35 = (np.abs(z) > 3.5).mean() * 100
    print(f"  {name:<20s} {kurt:>10.2f} {pct_3:>11.2f}% {pct_35:>11.2f}%")

print("\nBajo normalidad, la proporción teórica |z|>3 es 0.27%.")
print("Bajo t-Student(4), casi 15 veces superior: el criterio produce falsos positivos masivos.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.3 VISUALIZACIÓN COMPARATIVA DE COLAS
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(1, 1, figsize=(11, 5.5))
xx = np.linspace(-6, 6, 500)
ax.plot(xx, norm.pdf(xx), color=COL_TEAL, lw=2.4, label='Normal N(0,1)')
ax.plot(xx, t.pdf(xx, 4), color=COL_AMBER, lw=2.4, label='t-Student (df=4)')
ax.plot(xx, t.pdf(xx, 3), color=COL_RED, lw=2.4, label='t-Student (df=3)')
ax.axvline(3, color=COL_GRAY, linestyle='--', lw=1)
ax.axvline(-3, color=COL_GRAY, linestyle='--', lw=1)
ax.fill_between(xx, 0, norm.pdf(xx), where=(np.abs(xx) > 3),
                color=COL_TEAL, alpha=0.3)
ax.fill_between(xx, 0, t.pdf(xx, 4), where=(np.abs(xx) > 3),
                color=COL_AMBER, alpha=0.3)
ax.text(3.2, 0.02, '|z| > 3', fontsize=10, color=COL_GRAY)
ax.set_title('Comparación de colas: normal vs t-Student', fontsize=13)
ax.set_xlabel('z'); ax.set_ylabel('Densidad')
ax.set_ylim(0, 0.42)
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()

### 📝 Implicancia metodológica

La gráfica muestra visualmente por qué el criterio 3-sigma falla con colas pesadas: el área sombreada bajo la t-Student (amber) es varias veces mayor que bajo la normal (teal). Bajo colas pesadas, observar valores con |z| > 3 resulta perfectamente esperable y no debe interpretarse como evidencia de outlier. **Consecuencia práctica**: en finanzas, el criterio 3-sigma debe abandonarse en favor de métodos robustos (z-score MAD, cuantiles empíricos, tests específicos) o bien modelar explícitamente la distribución con cola pesada.

---
# PARTE II — Métodos univariados de detección

## 4. Regla IQR de Tukey ($k=1.5$ y $k=3$)

La formulación clásica de Tukey declara outlier a toda observación que caiga fuera del intervalo $[Q_1 - k \cdot \text{IQR},\; Q_3 + k \cdot \text{IQR}]$. El valor $k=1.5$ produce "outliers moderados" y $k=3$ produce "outliers extremos". Bajo normalidad, $k=1.5$ captura aproximadamente el 99,3% de la masa.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. DETECCIÓN POR IQR (k=1.5 y k=3)
# ══════════════════════════════════════════════════════════════

def detect_iqr(series, k=1.5):
    """Detección de outliers por rango intercuartílico (Tukey)."""
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - k * IQR, Q3 + k * IQR
    mask = (series < lower) | (series > upper)
    return mask, lower, upper


print("=" * 82)
print("REGLA DE TUKEY — Comparación k=1.5 (moderados) vs k=3 (extremos)")
print("=" * 82)
print(f"{'Variable':<28s} {'N(k=1.5)':>10s} {'% (k=1.5)':>11s} {'N(k=3)':>9s} {'% (k=3)':>10s}")
print("-" * 82)

for v in num_vars:
    x = df[v].astype(float)
    m15, _, _ = detect_iqr(x, k=1.5)
    m30, _, _ = detect_iqr(x, k=3.0)
    print(f"  {v:<26s} {m15.sum():>10d} {m15.mean()*100:>10.1f}%"
          f" {m30.sum():>9d} {m30.mean()*100:>9.1f}%")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.1 VISUALIZACIÓN: BOXPLOTS CON CERCAS DE TUKEY
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
key_vars = ['credit_amount', 'age', 'duration']

for ax, v in zip(axes, key_vars):
    data = df[v].astype(float)
    m15, lo15, hi15 = detect_iqr(data, k=1.5)
    m30, lo30, hi30 = detect_iqr(data, k=3.0)

    # Boxplot base
    bp = ax.boxplot(data, vert=True, patch_artist=True, widths=0.5,
                    boxprops=dict(facecolor=COL_TEAL, alpha=0.35),
                    medianprops=dict(color=COL_NAVY, linewidth=2),
                    flierprops=dict(marker='', alpha=0))
    # Puntos individuales
    jitter = np.random.uniform(-0.18, 0.18, size=len(data))
    ax.scatter(1 + jitter, data, c=COL_GRAY, s=6, alpha=0.25)
    ax.scatter((1 + jitter)[m15], data[m15], c=COL_AMBER, s=22,
               alpha=0.85, edgecolor='white', lw=0.3,
               label=f'k=1.5: {m15.sum()}')
    ax.scatter((1 + jitter)[m30], data[m30], c=COL_RED, s=34,
               alpha=0.95, edgecolor='black', lw=0.4,
               label=f'k=3: {m30.sum()}')
    ax.axhline(hi15, color=COL_AMBER, linestyle='--', lw=0.9)
    ax.axhline(hi30, color=COL_RED, linestyle='--', lw=0.9)
    ax.set_title(v); ax.legend(fontsize=8, loc='upper left')
    ax.set_xticks([])

plt.suptitle('Regla de Tukey: umbrales moderados (k=1.5) vs estrictos (k=3)',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 📝 Interpretación

La regla de Tukey produce clasificaciones binarias simples pero revela la **fragilidad del umbral** cuando la distribución subyacente es asimétrica. En `credit_amount`, la proporción de outliers con $k=1.5$ supera ampliamente el 0,7% esperable bajo normalidad, confirmando empíricamente la distribución de cola derecha pesada diagnosticada en la Parte I. En variables discretas de baja cardinalidad (`num_dependents`), el método produce un artefacto: clasifica como outlier a toda la categoría minoritaria, lo cual no refleja atipicidad genuina sino una limitación estructural del método.

---
## 5. IQR contextualizado por segmento

La regla global ignora la heterogeneidad interna de la población. Un ingreso o un monto de crédito pueden resultar "atípicos" respecto del total pero perfectamente típicos dentro de un segmento. Corregimos esta deficiencia mediante estratificación por variables categóricas relevantes.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. IQR CONTEXTUALIZADO POR SEGMENTO
# ══════════════════════════════════════════════════════════════

# Segmentación por tramo etario
df['tramo_edad'] = pd.cut(df['age'].astype(float),
                           bins=[18, 25, 35, 45, 55, 100],
                           labels=['18-25', '26-35', '36-45', '46-55', '56+'])

print("=" * 70)
print("IQR CONTEXTUALIZADO: credit_amount por tramo de edad")
print("=" * 70)
print(f"{'Tramo':<10s} {'N':>6s} {'Mediana':>10s} {'Q1':>10s} {'Q3':>10s} "
      f"{'Lím.sup':>10s} {'Outliers':>10s}")
print("-" * 75)

total_context = 0
mask_context_full = pd.Series(False, index=df.index)

for tramo in df['tramo_edad'].cat.categories:
    sub_idx = df['tramo_edad'] == tramo
    sub = df.loc[sub_idx, 'credit_amount'].astype(float)
    mask_ctx, lo, hi = detect_iqr(sub)
    # Mapear de vuelta al índice global
    mask_context_full.loc[sub_idx] = mask_ctx.values
    n_ctx = mask_ctx.sum()
    total_context += n_ctx
    print(f"  {tramo:<8s} {len(sub):>6d} {sub.median():>10.0f} "
          f"{sub.quantile(0.25):>10.0f} {sub.quantile(0.75):>10.0f} "
          f"{hi:>10.0f} {n_ctx:>10d}")

mask_global, _, _ = detect_iqr(df['credit_amount'].astype(float))
print(f"\nIQR global:         {mask_global.sum()} outliers")
print(f"IQR contextualizado: {total_context} outliers")
print(f"Coinciden:            {(mask_global & mask_context_full).sum()}")
print(f"Solo global:          {(mask_global & ~mask_context_full).sum()}")
print(f"Solo contextualizado: {(~mask_global & mask_context_full).sum()}")

### 📝 Interpretación

Los límites superiores difieren notoriamente entre tramos etarios: un crédito de 10.000 DM puede resultar típico en el segmento 46-55 años pero atípico en el segmento 18-25. El IQR contextualizado detecta casos que el método global pasa por alto (valores altos dentro de su segmento aunque modestos en términos absolutos) y a la inversa libera registros que el método global marcaba incorrectamente. Esta divergencia ilustra el principio: **la atipicidad es siempre relativa a un contexto poblacional**.

---
## 6. Z-score clásico vs z-score MAD — demostración del *masking*

El z-score clásico $z_i = (x_i - \bar{x})/s$ sufre de una vulnerabilidad circular: los propios outliers inflan $\bar{x}$ y $s$, reduciendo artificialmente los z-scores y enmascarando su propia presencia. Demostramos empíricamente este fenómeno contaminando una muestra limpia con un único valor extremo.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. DEMOSTRACIÓN EMPÍRICA DEL MASKING
# ══════════════════════════════════════════════════════════════

np.random.seed(RS)
# Muestra limpia de ingresos mensuales (en miles de pesos)
ingresos_limpios = np.random.gamma(shape=3.5, scale=12, size=50)
# Valor contaminado (error de captura: coma desplazada)
outlier_verdadero = 8_500.0
ingresos_contaminados = np.concatenate([ingresos_limpios, [outlier_verdadero]])


def z_clasico(x):
    return (x - np.mean(x)) / np.std(x, ddof=1)


def z_mad(x):
    med = np.median(x)
    mad = median_abs_deviation(x, scale=1.0)
    return 0.6745 * (x - med) / mad if mad > 0 else np.zeros_like(x)


print("=" * 72)
print("DEMOSTRACIÓN DEL MASKING — Muestra con UN outlier contaminante")
print("=" * 72)
print(f"Muestra: 50 ingresos en [~15, ~95] + 1 valor contaminado = 8.500")
print(f"\nEstadísticos con la contaminación:")
print(f"  Media:    {ingresos_contaminados.mean():>10.2f}")
print(f"  Desvío:   {ingresos_contaminados.std(ddof=1):>10.2f}")
print(f"  Mediana:  {np.median(ingresos_contaminados):>10.2f}")
print(f"  MAD:      {median_abs_deviation(ingresos_contaminados):>10.2f}")

z_cl = z_clasico(ingresos_contaminados)
z_m  = z_mad(ingresos_contaminados)

print(f"\nScore del outlier verdadero (8.500):")
print(f"  z-score clásico: {z_cl[-1]:>8.2f}   ¿Supera umbral 3?  {'✓' if abs(z_cl[-1]) > 3 else '✗ MASKING'}")
print(f"  z-score MAD:     {z_m[-1]:>8.2f}   ¿Supera umbral 3.5? {'✓' if abs(z_m[-1]) > 3.5 else '✗'}")

print(f"\nObservación: el z-score clásico ASIGNA {abs(z_cl[-1]):.2f} al outlier,")
print(f"valor insuficiente para superar el umbral convencional. El outlier")
print(f"se camufla dentro de su propia inflación de la desviación estándar.")
print(f"El z-score MAD lo detecta con score {abs(z_m[-1]):.1f}: completamente inmune al masking.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6.1 VISUALIZACIÓN DEL MASKING
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Panel A: histograma con el outlier
axes[0].hist(ingresos_limpios, bins=18, color=COL_TEAL, alpha=0.7,
             edgecolor='white', label='Muestra limpia (n=50)')
axes[0].scatter([outlier_verdadero], [0.3], s=350, marker='v',
                color=COL_RED, edgecolor='black', lw=1.2, zorder=5,
                label=f'Outlier: {outlier_verdadero:.0f}')
axes[0].set_xlabel('Ingreso mensual (miles de pesos)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Muestra contaminada — escala lineal')
axes[0].legend()

# Panel B: comparación de z-scores
idx = np.arange(len(ingresos_contaminados))
axes[1].scatter(idx[:-1], np.abs(z_cl[:-1]), s=30, color=COL_TEAL, alpha=0.6, label='|z| clásico (limpia)')
axes[1].scatter(idx[-1:], np.abs(z_cl[-1:]), s=160, color=COL_TEAL, marker='v', edgecolor='black', lw=1, label=f'|z| clásico (outlier) = {abs(z_cl[-1]):.2f}')
axes[1].scatter(idx[:-1], np.abs(z_m[:-1]),  s=30, color=COL_AMBER, alpha=0.6, label='|M| MAD (limpia)')
axes[1].scatter(idx[-1:], np.abs(z_m[-1:]),  s=160, color=COL_AMBER, marker='v', edgecolor='black', lw=1, label=f'|M| MAD (outlier) = {abs(z_m[-1]):.1f}')
axes[1].axhline(3,   color=COL_TEAL, linestyle='--', lw=1, label='Umbral z-clásico = 3')
axes[1].axhline(3.5, color=COL_AMBER, linestyle='--', lw=1, label='Umbral MAD = 3.5')
axes[1].set_yscale('log')
axes[1].set_xlabel('Índice de la observación')
axes[1].set_ylabel('|score| (escala log)')
axes[1].set_title('Masking: z-clásico vs z-MAD')
axes[1].legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

### 📝 Interpretación formal

El z-score clásico asigna al outlier un score $\approx 2.85$, valor que **no supera** el umbral convencional de 3. El *masking* se produce porque el propio outlier ha inflado la desviación estándar, elevándola desde ~10 a ~1180 (más de dos órdenes de magnitud). En contraste, el z-score MAD preserva la mediana y la MAD virtualmente inalteradas —ambas poseen punto de quiebre del 50%— y asigna al outlier un score superior a 600.

**Lección metodológica**: cuando se sospecha contaminación en los datos, jamás debe utilizarse el z-score clásico. El z-score modificado basado en MAD constituye la alternativa robusta estándar y debería usarse como primera línea de detección univariada.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6.2 APLICACIÓN AL GERMAN CREDIT
# ══════════════════════════════════════════════════════════════

def detect_zscore_mad(series, threshold=3.5):
    """Z-score modificado basado en MAD (Iglewicz & Hoaglin, 1993)."""
    med = np.median(series)
    mad = median_abs_deviation(series)
    if mad == 0:
        return pd.Series(False, index=series.index), pd.Series(0, index=series.index)
    scores = 0.6745 * (series - med) / mad
    mask = np.abs(scores) > threshold
    return mask, scores


def detect_zscore_classic(series, threshold=3.0):
    """Z-score clásico (no robusto)."""
    scores = (series - series.mean()) / series.std()
    mask = np.abs(scores) > threshold
    return mask, scores


print("=" * 82)
print("COMPARACIÓN: Z-SCORE CLÁSICO vs Z-SCORE MAD EN GERMAN CREDIT")
print("=" * 82)
print(f"{'Variable':<26s} {'Clásico |z|>3':>15s} {'MAD |M|>3.5':>15s} {'Diferencia':>12s}")
print("-" * 82)

for v in num_vars:
    x = df[v].astype(float)
    mc, _ = detect_zscore_classic(x, 3.0)
    mm, _ = detect_zscore_mad(x, 3.5)
    diff = mm.sum() - mc.sum()
    print(f"  {v:<24s} {mc.sum():>15d} {mm.sum():>15d} {diff:>+12d}")

---
## 7. Criterio de Chauvenet

Método clásico del siglo XIX, útil cuando se requiere justificar formalmente la eliminación de una observación extrema en muestras moderadas bajo hipótesis de normalidad. El umbral crece con el tamaño muestral: $|z_i| > \Phi^{-1}(1 - 1/(4n))$.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. CRITERIO DE CHAUVENET
# ══════════════════════════════════════════════════════════════

def chauvenet_threshold(n):
    """Umbral de z-score para el criterio de Chauvenet bajo normalidad."""
    return norm.ppf(1 - 1 / (4 * n))


def detect_chauvenet(series):
    """Detección por criterio de Chauvenet. Requiere normalidad aproximada."""
    n = len(series)
    thr = chauvenet_threshold(n)
    z = (series - series.mean()) / series.std()
    mask = np.abs(z) > thr
    return mask, thr


print("=" * 65)
print("CRITERIO DE CHAUVENET — Umbral dependiente de n")
print("=" * 65)
print(f"{'n':>10s} {'Umbral |z|':>15s}   Interpretación")
print("-" * 65)
for n in [10, 50, 100, 500, 1000, 5000, 10000]:
    thr = chauvenet_threshold(n)
    print(f"  {n:>8d} {thr:>13.3f}   " +
          f"Eliminar si P(|Z| > {thr:.2f}) < 1/(2·{n})")

print("\n" + "=" * 65)
print(f"APLICACIÓN AL GERMAN CREDIT (n = {len(df)})")
print("=" * 65)
thr = chauvenet_threshold(len(df))
print(f"Umbral Chauvenet: |z| > {thr:.3f}\n")
print(f"{'Variable':<26s} {'Outliers':>10s} {'% sobre n':>11s}")
print("-" * 50)
for v in num_vars:
    x = df[v].astype(float)
    mask, _ = detect_chauvenet(x)
    print(f"  {v:<24s} {mask.sum():>10d} {mask.mean()*100:>10.2f}%")

### 📝 Interpretación

El criterio de Chauvenet escala razonablemente con el tamaño muestral: en $n=10$ el umbral es apenas 1,96, mientras que en $n=10\,000$ asciende a 3,89. Esta escalabilidad refleja correctamente la intuición de que, en muestras grandes, observaciones con $|z| > 3$ resultan esperables solo por el volumen de datos. Sin embargo, el método **presupone normalidad** —hipótesis que ya rechazamos para estas variables—, por lo que sobre-detecta en presencia de colas pesadas.

---
## 8. Test de Grubbs

Test de hipótesis formal para un único outlier bajo normalidad de la muestra sin el candidato. Útil en control de calidad y análisis de laboratorio, menos apropiado para datos financieros (violación sistemática de la normalidad).

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. TEST DE GRUBBS
# ══════════════════════════════════════════════════════════════

def grubbs_test(x, alpha=0.05):
    """Test de Grubbs bilateral para un único outlier."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean_x = x.mean()
    std_x  = x.std(ddof=1)
    # Estadístico G: máxima desviación absoluta normalizada
    abs_dev = np.abs(x - mean_x)
    G = abs_dev.max() / std_x
    idx_outlier = int(np.argmax(abs_dev))
    # Valor crítico
    t_crit = t.ppf(1 - alpha / (2 * n), df=n - 2)
    G_crit = ((n - 1) / np.sqrt(n)) * np.sqrt(t_crit**2 / (n - 2 + t_crit**2))
    # p-valor aproximado
    t_stat = np.sqrt((n * (n - 2) * G**2) / ((n - 1)**2 - n * G**2))
    p_value = min(1.0, 2 * n * (1 - t.cdf(t_stat, df=n - 2)))
    return {
        'G': G, 'G_crit': G_crit, 'p_value': p_value,
        'reject_H0': G > G_crit, 'outlier_value': x[idx_outlier],
        'outlier_index': idx_outlier, 'n': n
    }


print("=" * 72)
print("TEST DE GRUBBS (α = 0.05) — Detección de un outlier bajo normalidad")
print("=" * 72)
print(f"{'Variable':<26s} {'G obs':>8s} {'G crít':>8s} {'p-valor':>10s} {'Rechazo H0?':>12s}")
print("-" * 72)

for v in num_vars:
    x = df[v].astype(float).values
    res = grubbs_test(x)
    flag = "✓" if res['reject_H0'] else "✗"
    print(f"  {v:<24s} {res['G']:>8.2f} {res['G_crit']:>8.2f} "
          f"{res['p_value']:>10.2e} {flag:>12s}")

print("\nAdvertencia: el test presupone normalidad de los demás datos (excluido el")
print("candidato). En variables no-gaussianas, los 'rechazos' pueden ser falsos.")

---
# PARTE III — Métodos multivariados

## 9. Distancia de Mahalanobis clásica

La distancia de Mahalanobis extiende el z-score al caso multivariado: considera tanto la escala de cada variable como la estructura de correlación. Una observación con valor típico en cada dimensión puede ser atípica en la distribución conjunta.

$$D_M(\mathbf{x}) = \sqrt{(\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})}$$

Bajo normalidad multivariada, $D_M^2 \sim \chi^2_p$, lo que provee umbrales teóricos. Su principal debilidad: $\hat{\boldsymbol{\Sigma}}$ se infla con outliers, produciendo *masking* en el espacio multivariado.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. MAHALANOBIS CLÁSICA
# ══════════════════════════════════════════════════════════════

# Submatriz para análisis multivariado (3 variables clave)
X_multi = df[['credit_amount', 'age', 'duration']].astype(float).values
p = X_multi.shape[1]

# Estimación clásica (no robusta)
cov_classic = EmpiricalCovariance().fit(X_multi)
mu_classic  = cov_classic.location_
d2_classic  = cov_classic.mahalanobis(X_multi)  # devuelve D^2
d_classic   = np.sqrt(d2_classic)

# Umbral chi-cuadrado al 99%
thr_099 = chi2.ppf(0.99, df=p)
mask_mahal_classic = d2_classic > thr_099

print("=" * 65)
print("DISTANCIA DE MAHALANOBIS CLÁSICA")
print("=" * 65)
print(f"Variables: credit_amount, age, duration  (p = {p})")
print(f"Umbral χ²({p}, 0.99) = {thr_099:.2f}")
print(f"\nOutliers detectados: {mask_mahal_classic.sum()} ({mask_mahal_classic.mean()*100:.1f}%)")
print(f"D² máximo: {d2_classic.max():.2f}")
print(f"D² mediano: {np.median(d2_classic):.2f}")

### 10. Mahalanobis robusta basada en MCD (Rousseeuw, 1985)

El estimador MCD busca el subconjunto de $h$ observaciones cuya matriz de covarianza tiene el menor determinante, y calcula la distancia de Mahalanobis usando las estimaciones sobre ese subconjunto "limpio". Posee un punto de quiebre cercano al 50%, inmune al masking que afecta a la versión clásica.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. MAHALANOBIS ROBUSTA (MCD)
# ══════════════════════════════════════════════════════════════

# FAST-MCD (Rousseeuw & Van Driessen, 1999)
mcd = MinCovDet(random_state=RS).fit(X_multi)
d2_robust = mcd.mahalanobis(X_multi)
d_robust  = np.sqrt(d2_robust)
mask_mahal_robust = d2_robust > thr_099

print("=" * 70)
print("MAHALANOBIS ROBUSTA — Minimum Covariance Determinant")
print("=" * 70)
print(f"Outliers detectados: {mask_mahal_robust.sum()} ({mask_mahal_robust.mean()*100:.1f}%)")
print(f"D² máximo robusto:   {d2_robust.max():.2f}")
print(f"D² clásico máximo:   {d2_classic.max():.2f}")
print(f"\n¿Cuántos outliers detectaron ambos métodos?")
print(f"  Clásico:                 {mask_mahal_classic.sum()}")
print(f"  Robusto (MCD):           {mask_mahal_robust.sum()}")
print(f"  Coinciden:               {(mask_mahal_classic & mask_mahal_robust).sum()}")
print(f"  Solo MCD (masked):       {(~mask_mahal_classic & mask_mahal_robust).sum()}")
print(f"  Solo clásico:            {(mask_mahal_classic & ~mask_mahal_robust).sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10.1 VISUALIZACIÓN COMPARATIVA: ELIPSES DE CONCENTRACIÓN
# ══════════════════════════════════════════════════════════════

def elipse_from_cov(ax, mean, cov, n_std=2.5, **kwargs):
    """Dibuja una elipse de concentración a n_std sqrt(chi2) de la media."""
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    e = Ellipse(xy=mean, width=w, height=h, angle=angle, fill=False, lw=2.2, **kwargs)
    ax.add_patch(e)
    return e


# Restringimos a 2D: credit_amount vs age
X_2d = df[['credit_amount', 'age']].astype(float).values
cov_cl_2d  = EmpiricalCovariance().fit(X_2d)
cov_rob_2d = MinCovDet(random_state=RS).fit(X_2d)

d2_cl_2d  = cov_cl_2d.mahalanobis(X_2d)
d2_rob_2d = cov_rob_2d.mahalanobis(X_2d)
thr_2d = chi2.ppf(0.99, df=2)

m_cl  = d2_cl_2d  > thr_2d
m_rob = d2_rob_2d > thr_2d

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

for ax, mask, mean, cov, title in zip(
        axes,
        [m_cl, m_rob],
        [cov_cl_2d.location_, cov_rob_2d.location_],
        [cov_cl_2d.covariance_, cov_rob_2d.covariance_],
        ['Mahalanobis clásica', 'Mahalanobis robusta (MCD)']):
    ax.scatter(X_2d[~mask, 0], X_2d[~mask, 1], c=COL_TEAL, s=18,
               alpha=0.45, label='Típicos')
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=COL_RED, s=40,
               alpha=0.9, edgecolor='black', lw=0.4,
               label=f'Outliers ({mask.sum()})')
    for n_std, alpha in [(np.sqrt(chi2.ppf(0.90, df=2)), 0.3),
                         (np.sqrt(chi2.ppf(0.99, df=2)), 1.0)]:
        elipse_from_cov(ax, mean, cov, n_std=n_std,
                        edgecolor=COL_NAVY, alpha=alpha)
    ax.scatter(*mean, c=COL_AMBER, s=120, marker='X',
               edgecolor='black', lw=1, zorder=6, label='Centroide')
    ax.set_xlabel('credit_amount'); ax.set_ylabel('age')
    ax.set_title(title); ax.legend(fontsize=9, loc='upper right')

plt.suptitle('Elipses de concentración al 90% y 99% — Clásica vs MCD robusta',
             fontweight='bold', fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

### 📝 Interpretación

La comparación visual exhibe el efecto del *masking* multivariado. Las elipses de la versión clásica son más amplias porque los propios outliers inflan la matriz de covarianza; como consecuencia, observaciones genuinamente atípicas escapan a la detección. Las elipses MCD, estimadas sobre el 75% más compacto de los datos, mantienen forma más ceñida y detectan correctamente los puntos periféricos.

**Regla operativa**: en presencia de sospecha de contaminación superior al 5%, la versión MCD debe preferirse sistemáticamente.

---
## 11. Local Outlier Factor (LOF)

A diferencia de los métodos basados en distancia al centroide, LOF compara la densidad local de una observación con la densidad de sus vecinos. Una observación en una región relativamente vacía dentro de su vecindario recibe LOF $\gg 1$, aun cuando globalmente no sea extrema. Resulta ideal cuando coexisten clusters de densidades heterogéneas.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. LOCAL OUTLIER FACTOR (LOF)
# ══════════════════════════════════════════════════════════════

# Normalizamos las variables para que LOF no se domine por escalas
scaler = StandardScaler()
X_multi_std = scaler.fit_transform(X_multi)

# LOF con k=20 vecinos
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False)
lof_pred = lof.fit_predict(X_multi_std)
lof_scores = -lof.negative_outlier_factor_  # valores altos = más atípico
mask_lof = lof_pred == -1

print("=" * 60)
print("LOCAL OUTLIER FACTOR (k=20, contamination=5%)")
print("=" * 60)
print(f"Outliers detectados: {mask_lof.sum()} ({mask_lof.mean()*100:.1f}%)")
print(f"LOF máximo: {lof_scores.max():.2f}")
print(f"LOF mediano: {np.median(lof_scores):.2f}")
print(f"\nLectura: LOF ≈ 1 indica densidad local similar a vecinos (observación típica).")
print(f"LOF >> 1 indica densidad local significativamente inferior (outlier).")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11.1 SENSIBILIDAD DE LOF AL PARÁMETRO k
# ══════════════════════════════════════════════════════════════

k_values = [5, 10, 20, 50, 100]
lof_results = {}
for k in k_values:
    lof_k = LocalOutlierFactor(n_neighbors=k, contamination=0.05)
    pred = lof_k.fit_predict(X_multi_std)
    lof_results[k] = {
        'mask': pred == -1,
        'scores': -lof_k.negative_outlier_factor_
    }

print("=" * 60)
print("SENSIBILIDAD DE LOF AL PARÁMETRO k")
print("=" * 60)
print(f"{'k':>6s} {'N outliers':>12s} {'%':>8s} {'LOF máx':>10s}")
print("-" * 50)
for k in k_values:
    r = lof_results[k]
    print(f"  {k:>4d} {r['mask'].sum():>12d} "
          f"{r['mask'].mean()*100:>7.1f}% {r['scores'].max():>10.2f}")

print("\nRecomendación de Breunig et al.: tomar LOF* = max_{k ∈ K} LOF_k")
print("como score final, evaluando un rango de k ∈ [10, 50].")

### 12. Isolation Forest (Liu, Ting & Zhou, 2008)

Algoritmo basado en una observación simple: los outliers resultan más fáciles de aislar mediante particiones aleatorias recursivas, requiriendo menos cortes que los puntos típicos. El score de anomalía se deriva de la longitud promedio del camino hasta la hoja. Escalable a datasets masivos ($O(T\psi \log \psi)$).

In [ ]:
# ══════════════════════════════════════════════════════════════
# 12. ISOLATION FOREST
# ══════════════════════════════════════════════════════════════

iso = IsolationForest(n_estimators=200, contamination=0.05,
                      random_state=RS, n_jobs=-1)
iso.fit(X_multi)
iso_pred   = iso.predict(X_multi)
iso_scores = -iso.score_samples(X_multi)  # invertir para que alto = atípico
mask_iso   = iso_pred == -1

print("=" * 60)
print("ISOLATION FOREST (T=200 árboles, contamination=5%)")
print("=" * 60)
print(f"Outliers detectados:  {mask_iso.sum()} ({mask_iso.mean()*100:.1f}%)")
print(f"Score máximo:         {iso_scores.max():.3f}")
print(f"Score mediano:        {np.median(iso_scores):.3f}")
print(f"Score típico outlier: {iso_scores[mask_iso].mean():.3f}")
print(f"Score típico normal:  {iso_scores[~mask_iso].mean():.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 12.1 DISTRIBUCIÓN DEL SCORE DE ANOMALÍA
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Histograma del score
axes[0].hist(iso_scores[~mask_iso], bins=40, color=COL_TEAL, alpha=0.6,
             edgecolor='white', label='Normales')
axes[0].hist(iso_scores[mask_iso], bins=20, color=COL_RED, alpha=0.85,
             edgecolor='white', label=f'Outliers ({mask_iso.sum()})')
thr_iso = iso_scores[mask_iso].min() if mask_iso.sum() else iso_scores.max()
axes[0].axvline(thr_iso, color=COL_NAVY, linestyle='--', lw=1.5,
                label=f'Umbral ≈ {thr_iso:.3f}')
axes[0].set_xlabel('Score de anomalía Isolation Forest')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del score')
axes[0].legend()

# Scatter credit_amount vs age coloreado por score
sc = axes[1].scatter(X_multi[:, 0], X_multi[:, 1], c=iso_scores,
                     cmap='RdYlGn_r', s=18, alpha=0.7, edgecolor='none')
axes[1].scatter(X_multi[mask_iso, 0], X_multi[mask_iso, 1], s=45,
                facecolor='none', edgecolor=COL_RED, lw=1.3,
                label=f'Outliers ({mask_iso.sum()})')
plt.colorbar(sc, ax=axes[1], label='Score')
axes[1].set_xlabel('credit_amount'); axes[1].set_ylabel('age')
axes[1].set_title('Score IForest en el plano (credit_amount × age)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### 13. DBSCAN como detector de outliers

Aunque diseñado para clustering, DBSCAN etiqueta naturalmente como "ruido" las observaciones que no pertenecen a ningún cluster denso. Esta etiqueta admite interpretación inmediata como outlier.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 13. DBSCAN
# ══════════════════════════════════════════════════════════════

# Selección de eps y min_samples
# Regla práctica: min_samples ≈ 2 * p (p = dimensionalidad)
db = DBSCAN(eps=0.7, min_samples=6, n_jobs=-1)
db_labels = db.fit_predict(X_multi_std)
mask_dbscan = db_labels == -1
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)

print("=" * 60)
print("DBSCAN (eps=0.7, min_samples=6, datos estandarizados)")
print("=" * 60)
print(f"Clusters detectados: {n_clusters}")
print(f"Puntos de ruido (outliers): {mask_dbscan.sum()} ({mask_dbscan.mean()*100:.1f}%)")

# Tamaños de los clusters
if n_clusters > 0:
    print("\nDistribución de tamaños de cluster:")
    sizes = pd.Series(db_labels[db_labels != -1]).value_counts().sort_index()
    for cluster_id, size in sizes.items():
        print(f"  Cluster {cluster_id}: {size} obs ({size/len(df)*100:.1f}%)")

### 14. One-Class SVM (Schölkopf et al., 2001)

Aprende una frontera de decisión no paramétrica que envuelve la región "normal" del espacio. El parámetro $\nu$ controla la fracción aproximada de outliers admitidos. Con kernel RBF resulta muy flexible pero sensible a hiperparámetros.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 14. ONE-CLASS SVM
# ══════════════════════════════════════════════════════════════

ocsvm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
ocsvm.fit(X_multi_std)
ocsvm_pred = ocsvm.predict(X_multi_std)
ocsvm_scores = -ocsvm.decision_function(X_multi_std)
mask_ocsvm = ocsvm_pred == -1

print("=" * 60)
print("ONE-CLASS SVM (kernel='rbf', ν=0.05)")
print("=" * 60)
print(f"Outliers detectados: {mask_ocsvm.sum()} ({mask_ocsvm.mean()*100:.1f}%)")
print(f"Parámetro ν = 0.05: fracción esperada de outliers")

### 15. Consenso entre métodos multivariados

La diversidad de algoritmos permite construir un **score de consenso**: observaciones marcadas por varios métodos simultáneamente son candidatos de alta confianza. Este principio (ensamble de detectores) reduce el riesgo de falsos positivos específicos de cada algoritmo.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 15. CONSENSO ENTRE MÉTODOS
# ══════════════════════════════════════════════════════════════

# Ensamblar máscaras
masks_multi = {
    'Mahalanobis clásica': mask_mahal_classic,
    'Mahalanobis MCD':     mask_mahal_robust,
    'LOF (k=20)':          mask_lof,
    'Isolation Forest':    mask_iso,
    'DBSCAN':              mask_dbscan,
    'One-Class SVM':       mask_ocsvm,
}

consenso = pd.DataFrame(masks_multi).astype(int)
consenso['votos'] = consenso.sum(axis=1)

print("=" * 70)
print("MATRIZ DE COINCIDENCIAS ENTRE MÉTODOS MULTIVARIADOS")
print("=" * 70)
print(f"{'Método':<28s} " + " ".join(f"{m[:10]:>11s}" for m in masks_multi))
print("-" * 95)

method_list = list(masks_multi.keys())
for i, mi in enumerate(method_list):
    row = [f"  {mi:<26s}"]
    for j, mj in enumerate(method_list):
        if i == j:
            row.append(f"{masks_multi[mi].sum():>11d}")
        else:
            inter = (masks_multi[mi] & masks_multi[mj]).sum()
            row.append(f"{inter:>11d}")
    print(" ".join(row))

print("\n" + "=" * 70)
print("DISTRIBUCIÓN DEL NÚMERO DE VOTOS POR REGISTRO")
print("=" * 70)
print(f"{'Votos':>7s} {'N registros':>12s} {'% sobre total':>14s}  Interpretación")
print("-" * 70)
for v in range(7):
    n = (consenso['votos'] == v).sum()
    interp = {
        0: "Normal (unanimidad negativa)",
        1: "Outlier débil",
        2: "Outlier moderado",
        3: "Outlier probable",
        4: "Outlier fuerte",
        5: "Outlier muy probable",
        6: "Outlier unánime (todos los métodos coinciden)"
    }.get(v, "")
    print(f"  {v:>5d} {n:>12d} {n/len(df)*100:>13.1f}%  {interp}")

# Outliers de consenso: votados por ≥ 4 métodos
mask_consenso = consenso['votos'] >= 4
print(f"\nCandidatos de alta confianza (≥4 votos): {mask_consenso.sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 15.1 VISUALIZACIÓN DEL CONSENSO
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel A: scatter coloreado por votos
sc = axes[0].scatter(X_multi[:, 0], X_multi[:, 1], c=consenso['votos'],
                     cmap='YlOrRd', s=25, alpha=0.75, edgecolor='none')
plt.colorbar(sc, ax=axes[0], label='Votos')
axes[0].set_xlabel('credit_amount'); axes[0].set_ylabel('age')
axes[0].set_title('Votos de detectores multivariados')

# Panel B: barplot de votos
vote_counts = consenso['votos'].value_counts().sort_index()
colors = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(vote_counts)))
axes[1].bar(vote_counts.index, vote_counts.values, color=colors,
            edgecolor='black', lw=0.4)
for x_b, y_b in zip(vote_counts.index, vote_counts.values):
    axes[1].text(x_b, y_b + 8, str(y_b), ha='center', fontsize=9)
axes[1].set_xlabel('Votos')
axes[1].set_ylabel('Registros')
axes[1].set_title('Distribución de consenso')

plt.tight_layout()
plt.show()

---
# PARTE IV — Método específico financiero: Ley de Benford

## 16. Ley de Benford (1938)

En datos financieros naturales que abarcan varios órdenes de magnitud (montos contables, transacciones, ingresos declarados), el **primer dígito significativo** $d \in \{1, \dots, 9\}$ se distribuye según:

$$P(D_1 = d) = \log_{10}\!\left(1 + \frac{1}{d}\right)$$

Desviaciones significativas respecto de este patrón constituyen evidencia estadística de **manipulación deliberada**. La administración tributaria argentina (AFIP) y auditores forenses incorporan tests de Benford en sus sistemas de selección de casos.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 16. LEY DE BENFORD — Prueba de adecuación
# ══════════════════════════════════════════════════════════════

def benford_expected(n=None):
    """Probabilidades teóricas del primer dígito."""
    digits = np.arange(1, 10)
    probs = np.log10(1 + 1/digits)
    if n is None:
        return digits, probs
    return digits, probs * n


def first_digit(x):
    """Extrae el primer dígito significativo de valores positivos."""
    x = np.abs(np.asarray(x, dtype=float))
    x = x[x > 0]
    # Escalar cada valor a [1, 10) y tomar su parte entera
    x_scaled = x / 10**np.floor(np.log10(x))
    return x_scaled.astype(int)


def benford_chi2_test(x):
    """Test chi-cuadrado de conformidad con la ley de Benford."""
    fd = first_digit(x)
    n = len(fd)
    observed = np.array([(fd == d).sum() for d in range(1, 10)])
    _, expected = benford_expected(n)
    chi2_stat = ((observed - expected)**2 / expected).sum()
    p_value = 1 - chi2.cdf(chi2_stat, df=8)
    return {
        'observed': observed, 'expected': expected,
        'chi2': chi2_stat, 'p_value': p_value, 'n': n
    }


# Aplicación a credit_amount (dataset natural)
res_credit = benford_chi2_test(df['credit_amount'].astype(float).values)

print("=" * 68)
print("TEST DE BENFORD — credit_amount (datos originales del dataset)")
print("=" * 68)
print(f"n = {res_credit['n']}  |  χ² = {res_credit['chi2']:.3f}  "
      f"|  p-valor = {res_credit['p_value']:.4f}")
print(f"\n{'Dígito':>8s} {'Observado':>12s} {'Esperado':>12s} {'Desvío %':>12s}")
print("-" * 50)
for d, obs, exp in zip(range(1, 10), res_credit['observed'], res_credit['expected']):
    desv = (obs - exp) / exp * 100
    print(f"  {d:>6d}  {obs:>12d}  {exp:>12.1f}  {desv:>+11.1f}%")

decision = "se conforma" if res_credit['p_value'] > 0.05 else "se desvía significativamente"
print(f"\nConclusión: la distribución {decision} de la Ley de Benford.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 16.1 SIMULACIÓN DE MANIPULACIÓN CONTABLE
# ══════════════════════════════════════════════════════════════

# Dataset sintético: 5000 transacciones bancarias genuinas (log-normales → Benford)
np.random.seed(RS)
transacciones_genuinas = np.random.lognormal(mean=7.5, sigma=1.8, size=5000)

# Dataset manipulado: mismo dataset pero con inyección de ~400 montos
# "cosméticos" just-below-threshold (montos típicos de estructuración: 9.500, 9.800, 9.900
# para evadir el umbral de reporte de 10.000)
transacciones_manipuladas = transacciones_genuinas.copy()
n_inyect = 400
idx_inyect = np.random.choice(len(transacciones_manipuladas), n_inyect, replace=False)
transacciones_manipuladas[idx_inyect] = np.random.choice(
    [9_500, 9_800, 9_900, 98_000, 990_000], size=n_inyect)

res_gen = benford_chi2_test(transacciones_genuinas)
res_man = benford_chi2_test(transacciones_manipuladas)

print("=" * 70)
print("COMPARACIÓN — Dataset genuino vs dataset manipulado")
print("=" * 70)
print(f"{'Dataset':<18s} {'n':>7s} {'χ²':>10s} {'p-valor':>12s} {'Conclusión':>28s}")
print("-" * 78)
for name, res in [('Genuino',     res_gen), ('Manipulado', res_man)]:
    concl = "Conforme a Benford" if res['p_value'] > 0.05 else "⚠ Desviación significativa"
    print(f"  {name:<16s} {res['n']:>7d} {res['chi2']:>10.2f} "
          f"{res['p_value']:>12.4f}  {concl:>28s}")

print(f"\nLa inyección de solo {n_inyect/len(transacciones_manipuladas)*100:.1f}% de montos")
print("manipulados (just-below-threshold) produce un rechazo inequívoco de Benford.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 16.2 VISUALIZACIÓN DE BENFORD
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for ax, res, title, color_obs in zip(axes,
                                      [res_gen, res_man],
                                      ['Genuino (log-normal)', 'Manipulado (structuring)'],
                                      [COL_TEAL, COL_RED]):
    digits = np.arange(1, 10)
    bar_obs = ax.bar(digits - 0.2, res['observed'] / res['n'] * 100, width=0.4,
                     color=color_obs, alpha=0.85, edgecolor='white',
                     label='Observado')
    bar_exp = ax.bar(digits + 0.2, res['expected'] / res['n'] * 100, width=0.4,
                     color=COL_GRAY, alpha=0.7, edgecolor='white',
                     label='Benford teórico')
    ax.set_xticks(digits)
    ax.set_xlabel('Primer dígito significativo')
    ax.set_ylabel('Frecuencia (%)')
    ax.set_title(f"{title}\nχ² = {res['chi2']:.2f}  |  p = {res['p_value']:.4f}")
    ax.legend()

plt.suptitle('Ley de Benford — Detección de manipulación contable',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 📝 Interpretación

En el dataset genuino, las frecuencias observadas se alinean con las predicciones de Benford: el dígito 1 domina con ~30%, el 2 con ~17,6%, y la frecuencia decae monotónicamente hasta el 9 con ~4,6%. En el dataset manipulado, la inyección de montos "cosméticos" (9.500, 9.800, 98.000, etc.) produce una **sobrerrepresentación anormal del dígito 9** y una subrepresentación del dígito 1. El test chi-cuadrado rechaza la conformidad con Benford con contundencia.

**Aplicabilidad**: la ley requiere que los datos abarquen varios órdenes de magnitud y no estén sujetos a mínimos o máximos artificiales. No se aplica a DNIs, códigos postales o precios estandarizados.

---
# PARTE V — Impacto estadístico de los outliers

## 17. Sesgo en media, varianza y correlación

Demostramos empíricamente cómo un único outlier puede distorsionar estimadores clásicos, y cómo las alternativas robustas preservan la señal subyacente.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 17. IMPACTO SOBRE ESTIMADORES CLÁSICOS VS ROBUSTOS
# ══════════════════════════════════════════════════════════════

np.random.seed(RS)
# Muestra gaussiana limpia
n = 200
x_clean = np.random.normal(loc=50, scale=10, size=n)
y_clean = 2 * x_clean + 5 + np.random.normal(0, 5, size=n)

# Contaminación: 5 outliers extremos
n_out = 5
x_cont = np.append(x_clean, [120, 130, 125, 135, 128])
y_cont = np.append(y_clean, [30, 25, 20, 15, 10])   # outliers verticales + leverage

print("=" * 70)
print("IMPACTO DE 5 OUTLIERS SOBRE ESTIMADORES (muestra original n=200)")
print("=" * 70)
print(f"{'Estadístico':<28s} {'Limpia':>12s} {'Contaminada':>14s} {'Cambio %':>12s}")
print("-" * 70)
estadisticos = [
    ('Media (x)',            x_clean.mean(),            x_cont.mean()),
    ('Mediana (x)',           np.median(x_clean),         np.median(x_cont)),
    ('Desvío (x)',           x_clean.std(ddof=1),        x_cont.std(ddof=1)),
    ('MAD (x)',              median_abs_deviation(x_clean), median_abs_deviation(x_cont)),
    ('Correlación Pearson',   np.corrcoef(x_clean, y_clean)[0,1],  np.corrcoef(x_cont, y_cont)[0,1]),
    ('Correlación Spearman',  stats.spearmanr(x_clean, y_clean).correlation,
                              stats.spearmanr(x_cont, y_cont).correlation),
]
for name, a, b in estadisticos:
    change = (b - a) / abs(a) * 100 if a != 0 else np.nan
    print(f"  {name:<26s} {a:>12.3f} {b:>14.3f} {change:>+11.1f}%")

print("\nObservación clave: la media y el desvío se distorsionan severamente.")
print("Mediana, MAD y correlación de Spearman resisten al 2.5% de contaminación.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 18. LEVERAGE POINTS Y DISTANCIA DE COOK EN REGRESIÓN OLS
# ══════════════════════════════════════════════════════════════

# Reconstruimos regresión OLS y evaluamos influencia
X_ols = sm.add_constant(x_cont)
y_ols = y_cont
model_ols = sm.OLS(y_ols, X_ols).fit()
influence = model_ols.get_influence()

leverage       = influence.hat_matrix_diag
std_resid      = influence.resid_studentized_internal
cook_distance  = influence.cooks_distance[0]

# Identificar los puntos más influyentes
n_total = len(x_cont)
thr_leverage = 2 * (X_ols.shape[1]) / n_total
thr_cook     = 4 / n_total

print("=" * 70)
print("DIAGNÓSTICO DE INFLUENCIA EN REGRESIÓN OLS")
print("=" * 70)
print(f"Umbral leverage (2p/n):    {thr_leverage:.4f}")
print(f"Umbral Cook (4/n):          {thr_cook:.4f}")
print(f"\nPuntos con alto leverage:  {(leverage > thr_leverage).sum()}")
print(f"Puntos con alta Cook:      {(cook_distance > thr_cook).sum()}")
print(f"Bad leverage points:        {((leverage > thr_leverage) & (np.abs(std_resid) > 2)).sum()}")

# Top 5 observaciones más influyentes
top_cook = np.argsort(cook_distance)[-5:][::-1]
print(f"\nTop 5 observaciones por Cook's D:")
print(f"  {'idx':>5s} {'x':>10s} {'y':>10s} {'leverage':>12s} {'residuo':>10s} {'Cook':>10s}")
for idx in top_cook:
    print(f"  {idx:>5d} {x_cont[idx]:>10.2f} {y_cont[idx]:>10.2f} "
          f"{leverage[idx]:>12.4f} {std_resid[idx]:>10.2f} {cook_distance[idx]:>10.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 18.1 VISUALIZACIÓN LEVERAGE + COOK
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

# Panel A: scatter original con las dos rectas (clean vs contaminated)
# Modelo limpio
X_clean_ols = sm.add_constant(x_clean)
model_clean = sm.OLS(y_clean, X_clean_ols).fit()
# Modelo contaminado
xx = np.linspace(20, 140, 100)
yy_clean = model_clean.params[0] + model_clean.params[1] * xx
yy_cont  = model_ols.params[0]   + model_ols.params[1]   * xx

axes[0].scatter(x_clean, y_clean, c=COL_TEAL, s=18, alpha=0.5, label='Limpia')
axes[0].scatter(x_cont[-n_out:], y_cont[-n_out:], c=COL_RED, s=90,
                marker='*', edgecolor='black', lw=0.6, label='Contaminados')
axes[0].plot(xx, yy_clean, color=COL_TEAL, lw=2.2, label='OLS limpia')
axes[0].plot(xx, yy_cont,  color=COL_RED,  lw=2.2, label='OLS contaminada')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_title('Efecto de los outliers sobre OLS')
axes[0].legend(fontsize=8)

# Panel B: leverage vs residuo (residual-leverage plot)
axes[1].scatter(leverage, std_resid, s=25, c=COL_TEAL, alpha=0.6)
axes[1].axhline(2, color=COL_AMBER, linestyle='--', lw=1, label='|resid. stud.| = 2')
axes[1].axhline(-2, color=COL_AMBER, linestyle='--', lw=1)
axes[1].axvline(thr_leverage, color=COL_RED, linestyle='--', lw=1,
                label=f'2p/n = {thr_leverage:.3f}')
for idx in top_cook:
    axes[1].scatter(leverage[idx], std_resid[idx], s=120, marker='o',
                    facecolor='none', edgecolor=COL_RED, lw=1.3)
    axes[1].annotate(str(idx), (leverage[idx], std_resid[idx]),
                     fontsize=8, xytext=(5, 5), textcoords='offset points')
axes[1].set_xlabel('Leverage $h_{ii}$')
axes[1].set_ylabel('Residuo studentizado')
axes[1].set_title('Residual-Leverage plot')
axes[1].legend(fontsize=8)

# Panel C: stem plot de la Cook's distance
axes[2].stem(np.arange(n_total), cook_distance, linefmt=COL_TEAL, markerfmt='o', basefmt=' ')
axes[2].axhline(thr_cook, color=COL_RED, linestyle='--', lw=1, label=f'4/n = {thr_cook:.3f}')
axes[2].axhline(1, color=COL_AMBER, linestyle='--', lw=1, label="D = 1 (crítico)")
axes[2].set_xlabel('Índice')
axes[2].set_ylabel("Cook's distance")
axes[2].set_title("Cook's distance por observación")
axes[2].legend(fontsize=8)

plt.suptitle('Diagnóstico formal de outliers en regresión: leverage y Cook',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 📝 Interpretación

El panel A visualiza el impacto catastrófico que los 5 puntos contaminantes ejercen sobre la recta OLS: la pendiente estimada se reduce sustancialmente porque los outliers combinan alto *leverage* (x extremo) con residuos verticales grandes —son "bad leverage points" en la nomenclatura de Rousseeuw. El panel B exhibe el gráfico residual-leverage con los umbrales de diagnóstico. El panel C muestra la distancia de Cook, donde los outliers destacan inequívocamente. Este diagnóstico formal constituye un paso obligatorio en la validación regulatoria de modelos internos de riesgo.

---
# PARTE VI — Estrategias de tratamiento

## 19. Transformaciones: log, Box-Cox, Yeo-Johnson

Las transformaciones de potencia a menudo convierten distribuciones asimétricas con colas pesadas en distribuciones aproximadamente gaussianas, sin necesidad de eliminar ni winsorizar observaciones. Esta vía preserva toda la información original.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 19. TRANSFORMACIONES DE POTENCIA
# ══════════════════════════════════════════════════════════════

ca = df['credit_amount'].astype(float).values

# T1: logarítmica
ca_log = np.log1p(ca)

# T2: Box-Cox (requiere estricta positividad)
ca_bc, lambda_bc = boxcox(ca + 1)

# T3: Yeo-Johnson (admite valores no-positivos)
pt = PowerTransformer(method='yeo-johnson', standardize=False)
ca_yj = pt.fit_transform(ca.reshape(-1, 1)).flatten()
lambda_yj = pt.lambdas_[0]

print("=" * 75)
print("TRANSFORMACIONES DE CREDIT_AMOUNT")
print("=" * 75)
print(f"{'Transformación':<24s} {'Asimetría':>11s} {'Curtosis':>10s} "
      f"{'Shapiro p':>13s} {'λ':>10s}")
print("-" * 75)
for name, series, lam in [
    ('Original',           ca,    None),
    ('log(1+x)',           ca_log, 0.0),
    ('Box-Cox',            ca_bc, lambda_bc),
    ('Yeo-Johnson',        ca_yj, lambda_yj)
]:
    skew_v = stats.skew(series)
    kurt_v = stats.kurtosis(series, fisher=False)
    sp = shapiro(series[:5000])[1]
    lam_str = "—" if lam is None else f"{lam:.3f}"
    print(f"  {name:<22s} {skew_v:>11.3f} {kurt_v:>10.2f} {sp:>13.2e} {lam_str:>10s}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 19.1 VISUALIZACIÓN DE LAS TRANSFORMACIONES
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 4, figsize=(17, 9))
transformations = [
    ('Original', ca, COL_TEAL),
    ('log(1+x)', ca_log, COL_AMBER),
    ('Box-Cox', ca_bc, COL_PURPLE),
    ('Yeo-Johnson', ca_yj, COL_GREEN),
]

for j, (name, series, color) in enumerate(transformations):
    # Histograma
    axes[0, j].hist(series, bins=45, color=color, alpha=0.75, edgecolor='white')
    axes[0, j].axvline(np.mean(series), color=COL_RED, linestyle='--',
                       lw=1.3, label=f'Media={np.mean(series):.2f}')
    axes[0, j].axvline(np.median(series), color=COL_NAVY, linestyle='--',
                       lw=1.3, label=f'Mediana={np.median(series):.2f}')
    axes[0, j].set_title(f'{name}\nSkew={stats.skew(series):.2f}, '
                         f'Kurt={stats.kurtosis(series, fisher=False):.1f}')
    axes[0, j].legend(fontsize=7)

    # Q-Q plot
    stats.probplot(series, dist="norm", plot=axes[1, j])
    axes[1, j].get_lines()[0].set_markerfacecolor(color)
    axes[1, j].get_lines()[0].set_markeredgecolor(color)
    axes[1, j].get_lines()[0].set_markersize(3.5)
    axes[1, j].get_lines()[1].set_color(COL_RED)
    axes[1, j].get_lines()[1].set_linewidth(1.6)
    axes[1, j].set_title(f'Q-Q plot: {name}')

plt.suptitle('Comparación de transformaciones sobre credit_amount',
             fontweight='bold', fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

### 📝 Interpretación

Las tres transformaciones reducen sustancialmente la asimetría y la curtosis de `credit_amount`. La transformación logarítmica simple produce una mejora notable pero no óptima. Box-Cox y Yeo-Johnson estiman el parámetro $\lambda$ por máxima verosimilitud y alcanzan niveles de normalidad superiores; Yeo-Johnson presenta la ventaja adicional de admitir valores no-positivos. En los Q-Q plots, las series transformadas se alinean mucho mejor con la recta de referencia gaussiana, habilitando el uso de procedimientos estadísticos paramétricos.

---
## 20. Regresión robusta de Huber vs OLS

Retomamos el ejemplo de regresión contaminada de la Sección 18 y contrastamos la recta OLS con la recta estimada mediante regresión M de Huber, que ofrece 95% de eficiencia bajo normalidad y robustez ante contaminación moderada.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 20. REGRESIÓN ROBUSTA DE HUBER
# ══════════════════════════════════════════════════════════════

# Modelos sobre datos contaminados
X_ols = sm.add_constant(x_cont)
model_ols = sm.OLS(y_cont, X_ols).fit()
model_rlm_huber = RLM(y_cont, X_ols, M=HuberT()).fit()
model_rlm_tukey = RLM(y_cont, X_ols, M=TukeyBiweight()).fit()

# Modelo "oracle" sobre datos limpios (referencia)
X_clean_ols = sm.add_constant(x_clean)
model_oracle = sm.OLS(y_clean, X_clean_ols).fit()

print("=" * 72)
print("COMPARACIÓN DE REGRESIONES SOBRE DATOS CONTAMINADOS")
print("=" * 72)
print(f"{'Modelo':<26s} {'β₀ (intercepto)':>18s} {'β₁ (pendiente)':>18s}")
print("-" * 72)
print(f"  {'Oracle (datos limpios)':<26s} {model_oracle.params[0]:>18.3f} "
      f"{model_oracle.params[1]:>18.3f}")
print(f"  {'OLS (contaminada)':<26s} {model_ols.params[0]:>18.3f} "
      f"{model_ols.params[1]:>18.3f}")
print(f"  {'Huber M (contaminada)':<26s} {model_rlm_huber.params[0]:>18.3f} "
      f"{model_rlm_huber.params[1]:>18.3f}")
print(f"  {'Tukey biweight (contam.)':<26s} {model_rlm_tukey.params[0]:>18.3f} "
      f"{model_rlm_tukey.params[1]:>18.3f}")

print(f"\nValores verdaderos: β₀ = 5, β₁ = 2")
print(f"Los estimadores M robustos recuperan valores próximos al oráculo,")
print(f"inmunes a los 5 puntos contaminantes que desplazan la recta OLS.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 20.1 VISUALIZACIÓN COMPARATIVA
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(1, 1, figsize=(11, 6.5))

ax.scatter(x_clean, y_clean, c=COL_TEAL, s=18, alpha=0.55, label='Limpios (n=200)')
ax.scatter(x_cont[-n_out:], y_cont[-n_out:], c=COL_RED, s=130,
           marker='*', edgecolor='black', lw=0.8, label=f'Contaminantes ({n_out})')

xx = np.linspace(20, 140, 150)
ax.plot(xx, model_oracle.params[0] + model_oracle.params[1]*xx,
        color=COL_NAVY, lw=2.6, linestyle='-', label=f'Oracle (β₁={model_oracle.params[1]:.2f})')
ax.plot(xx, model_ols.params[0] + model_ols.params[1]*xx,
        color=COL_RED, lw=2.4, linestyle='--', label=f'OLS (β₁={model_ols.params[1]:.2f})')
ax.plot(xx, model_rlm_huber.params[0] + model_rlm_huber.params[1]*xx,
        color=COL_AMBER, lw=2.4, linestyle='-.', label=f'Huber (β₁={model_rlm_huber.params[1]:.2f})')
ax.plot(xx, model_rlm_tukey.params[0] + model_rlm_tukey.params[1]*xx,
        color=COL_PURPLE, lw=2.4, linestyle=':', label=f'Tukey bw (β₁={model_rlm_tukey.params[1]:.2f})')

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Regresión robusta: OLS vs M-estimadores bajo contaminación',
             fontsize=13)
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

### 📝 Interpretación

La gráfica ilustra con claridad geométrica la ventaja de los M-estimadores: las rectas de Huber y Tukey-biweight prácticamente se superponen con el oráculo (la recta ajustada sobre los datos limpios), mientras que OLS se desvía sustancialmente. La función de pérdida de Huber crece linealmente en las colas —no cuadráticamente—, atenuando el peso de los puntos con residuos grandes sin eliminarlos. Tukey-biweight va un paso más allá: asigna peso nulo a observaciones con residuos muy grandes, recuperando el 95% de eficiencia bajo normalidad y un punto de quiebre cercano al 50%.

**Regla práctica para modelos financieros**: si existe sospecha razonable de contaminación, Huber debe considerarse como alternativa predeterminada a OLS. El costo computacional es marginal y la robustez adicional resulta frecuentemente determinante.

---
# PARTE VII — Evaluación en scoring crediticio

## 21. Impacto de cada estrategia sobre el AUC de un modelo de scoring

Cerramos evaluando cuantitativamente cómo cada estrategia de tratamiento de outliers modifica la capacidad predictiva de un modelo logístico de scoring sobre la variable objetivo (good/bad credit).

In [ ]:
# ══════════════════════════════════════════════════════════════
# 21. IMPACTO EN AUC: 5-FOLD CROSS-VALIDATION
# ══════════════════════════════════════════════════════════════

X_base = df[num_vars].astype(float).copy()
ca = X_base['credit_amount'].values

# Preparar variables transformadas
ca_winsor = pd.Series(winsorize(ca, limits=[0.01, 0.01]), index=X_base.index)
ca_log2    = np.log1p(ca)
ca_bc2, _  = boxcox(ca + 1)
ca_yj2     = PowerTransformer(method='yeo-johnson').fit_transform(
    ca.reshape(-1, 1)).flatten()

# Máscara de IQR para eliminación y flag
mask_iqr_global, _, _ = detect_iqr(X_base['credit_amount'])

# Definir estrategias
strategies = {
    '(0) Sin tratamiento':     X_base.copy(),
    '(1) Eliminar IQR':        X_base[~mask_iqr_global].copy(),
    '(2) Winsorizar P1/P99':   X_base.assign(credit_amount=ca_winsor),
    '(3) log(1+x)':            X_base.assign(credit_amount=ca_log2),
    '(4) Box-Cox':             X_base.assign(credit_amount=ca_bc2),
    '(5) Yeo-Johnson':         X_base.assign(credit_amount=ca_yj2),
    '(6) Flag de outlier':     X_base.assign(
        is_outlier_ca=mask_iqr_global.astype(int)),
}

# Baseline para delta
baseline_auc = cross_val_score(
    LogisticRegression(max_iter=2000, random_state=RS),
    X_base, y, cv=5, scoring='roc_auc').mean()

print("=" * 65)
print("IMPACTO EN AUC — Validación cruzada 5-fold (baseline = sin trat.)")
print("=" * 65)
print(f"{'Estrategia':<30s} {'AUC':>10s} {'Δ vs base':>12s}")
print("-" * 55)

for name, X_strat in strategies.items():
    y_strat = y if name != '(1) Eliminar IQR' else y[~mask_iqr_global.values]
    auc = cross_val_score(
        LogisticRegression(max_iter=2000, random_state=RS),
        X_strat, y_strat, cv=5, scoring='roc_auc').mean()
    delta = auc - baseline_auc
    flag = " ←" if delta > 0.005 else ""
    print(f"  {name:<28s} {auc:>10.4f} {delta:>+12.4f}{flag}")

print(f"\nBaseline AUC: {baseline_auc:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 21.1 VISUALIZACIÓN DE IMPACTOS
# ══════════════════════════════════════════════════════════════

strategy_names = list(strategies.keys())
aucs = []
for name, X_strat in strategies.items():
    y_strat = y if name != '(1) Eliminar IQR' else y[~mask_iqr_global.values]
    auc = cross_val_score(
        LogisticRegression(max_iter=2000, random_state=RS),
        X_strat, y_strat, cv=5, scoring='roc_auc').mean()
    aucs.append(auc)

fig, ax = plt.subplots(1, 1, figsize=(12, 5.5))
colors = [COL_GRAY if 'Sin' in n else COL_TEAL for n in strategy_names]
bars = ax.barh(strategy_names, aucs, color=colors, edgecolor='black', lw=0.4)
ax.axvline(baseline_auc, color=COL_RED, linestyle='--', lw=1.3,
           label=f'Baseline = {baseline_auc:.4f}')
for b, auc in zip(bars, aucs):
    ax.text(auc + 0.001, b.get_y() + b.get_height()/2,
            f'{auc:.4f}', va='center', fontsize=9)
ax.set_xlabel('AUC-ROC (CV 5-fold)')
ax.set_xlim(min(aucs) - 0.008, max(aucs) + 0.015)
ax.set_title('Impacto de cada estrategia sobre el AUC del scoring')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 📝 Interpretación

Los resultados empíricos confirman la doctrina del apunte:

- **Eliminar outliers** degrada el AUC: se pierde información relevante (los montos elevados tienen relación con la tasa de default).
- **Winsorizar** preserva el número de registros y suele mejorar ligeramente el AUC al atenuar el efecto desproporcionado de los valores extremos sobre los coeficientes logísticos.
- **Log, Box-Cox y Yeo-Johnson** ofrecen mejoras similares: comprimen la cola derecha y aproximan la variable a una escala donde la regresión lineal (sobre el logit) modela mejor la relación.
- **Agregar el flag de outlier** como variable adicional puede aportar información predictiva: el estatus de "atípico" puede correlacionar con la tasa de default.

**Regla de oro en scoring financiero**: antes de decidir el tratamiento, verificar el origen del outlier. En modelos regulatorios sujetos a validación (BCRA, IRB bajo Basilea), la eliminación de registros requiere documentación explícita de auditoría.

---
## 22. Protocolo operativo integral

Cerramos el notebook con una función que consolida las seis etapas del protocolo del apunte: exploración visual, detección univariada, detección multivariada, investigación, decisión de tratamiento y validación post-tratamiento.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 22. PROTOCOLO INTEGRAL CONSOLIDADO
# ══════════════════════════════════════════════════════════════

def protocolo_outliers(df_in, num_vars, contamination=0.05, verbose=True):
    """
    Protocolo integral de detección de outliers en 3 fases:
      1. Detección univariada (IQR + z-MAD)
      2. Detección multivariada (Mahalanobis MCD + Isolation Forest + LOF)
      3. Consenso (voto mayoritario)
    Retorna un DataFrame con los scores y máscaras de cada método.
    """
    X = df_in[num_vars].astype(float).copy()
    X_std = StandardScaler().fit_transform(X)
    n = len(X)

    resultados = pd.DataFrame(index=X.index)

    # ─────── 1. UNIVARIADA ───────
    if verbose:
        print("【Fase 1: Detección univariada】")
    votos_uni = pd.Series(0, index=X.index)
    for v in num_vars:
        m_iqr, _, _ = detect_iqr(X[v])
        m_mad, _    = detect_zscore_mad(X[v])
        votos_uni += m_iqr.astype(int) + m_mad.astype(int)
        resultados[f'iqr_{v}']  = m_iqr.astype(int)
        resultados[f'mad_{v}']  = m_mad.astype(int)
    resultados['votos_univariada'] = votos_uni

    # ─────── 2. MULTIVARIADA ───────
    if verbose:
        print("【Fase 2: Detección multivariada】")
    mcd = MinCovDet(random_state=RS).fit(X.values)
    d2  = mcd.mahalanobis(X.values)
    thr = chi2.ppf(0.99, df=len(num_vars))
    resultados['mcd_outlier'] = (d2 > thr).astype(int)
    resultados['mcd_dist']    = np.sqrt(d2)

    iso_m = IsolationForest(contamination=contamination, random_state=RS,
                            n_estimators=200, n_jobs=-1).fit(X.values)
    resultados['iso_outlier'] = (iso_m.predict(X.values) == -1).astype(int)
    resultados['iso_score']   = -iso_m.score_samples(X.values)

    lof_m = LocalOutlierFactor(n_neighbors=20, contamination=contamination)
    resultados['lof_outlier'] = (lof_m.fit_predict(X_std) == -1).astype(int)
    resultados['lof_score']   = -lof_m.negative_outlier_factor_

    # ─────── 3. CONSENSO ───────
    if verbose:
        print("【Fase 3: Consenso multivariado】")
    resultados['votos_multi']  = (
        resultados['mcd_outlier'] +
        resultados['iso_outlier'] +
        resultados['lof_outlier']
    )
    resultados['outlier_final'] = (
        (resultados['votos_multi'] >= 2) |
        (resultados['votos_univariada'] >= 3)
    ).astype(int)

    if verbose:
        print(f"\n→ Detectados {resultados['outlier_final'].sum()} outliers de consenso "
              f"({resultados['outlier_final'].mean()*100:.1f}%)")
        print(f"  • Consenso univariado (≥3 votos):  {(resultados['votos_univariada'] >= 3).sum()}")
        print(f"  • Consenso multivariado (≥2/3):    {(resultados['votos_multi'] >= 2).sum()}")
        print(f"  • Coincidencia ambos criterios:    "
              f"{((resultados['votos_univariada'] >= 3) & (resultados['votos_multi'] >= 2)).sum()}")

    return resultados


# Aplicar al German Credit
resultados = protocolo_outliers(df, num_vars)
print("\n" + "=" * 55)
print("DISTRIBUCIÓN DE VOTOS DEL PROTOCOLO INTEGRAL")
print("=" * 55)
print(resultados[['votos_univariada', 'votos_multi', 'outlier_final']].describe().round(2))

In [ ]:
# ══════════════════════════════════════════════════════════════
# 22.1 PERFIL DE OUTLIERS DE CONSENSO
# ══════════════════════════════════════════════════════════════

mask_final = resultados['outlier_final'] == 1

print("=" * 68)
print(f"PERFIL DE LOS {mask_final.sum()} OUTLIERS DE CONSENSO")
print("=" * 68)
print(f"{'Variable':<28s} {'Mediana normal':>16s} {'Mediana outlier':>18s}")
print("-" * 68)
for v in num_vars:
    med_normal  = df.loc[~mask_final, v].astype(float).median()
    med_outlier = df.loc[mask_final, v].astype(float).median()
    print(f"  {v:<26s} {med_normal:>16.2f} {med_outlier:>18.2f}")

print(f"\nTasa de default en outliers:  {y[mask_final.values].mean()*100:.1f}%")
print(f"Tasa de default en normales:  {y[~mask_final.values].mean()*100:.1f}%")
print(f"Ratio de riesgo:              {y[mask_final.values].mean()/y[~mask_final.values].mean():.2f}×")

print("\nLectura crítica: los outliers de consenso presentan una tasa de default")
print("diferenciada. Eliminarlos descartaría precisamente la subpoblación")
print("más informativa para el modelo. El tratamiento correcto es preservarlos")
print("con una marca explícita o segmentar el modelo.")

---
## 23. Síntesis: mapa de decisión del analista

In [ ]:
# ══════════════════════════════════════════════════════════════
# 23. SÍNTESIS FINAL
# ══════════════════════════════════════════════════════════════

sintesis = [
    "╔══════════════════════════════════════════════════════════════════════╗",
    "║   SÍNTESIS — DETECCIÓN Y TRATAMIENTO DE OUTLIERS EN FINANZAS         ║",
    "╠══════════════════════════════════════════════════════════════════════╣",
    "║                                                                      ║",
    "║  【 FUNDAMENTOS 】                                                   ║",
    "║  · Verificar normalidad (Shapiro, Jarque-Bera) antes de actuar      ║",
    "║  · En finanzas, las colas son SIEMPRE pesadas: olvidar 3-sigma      ║",
    "║                                                                      ║",
    "║  【 DETECCIÓN UNIVARIADA 】                                          ║",
    "║  · IQR (Tukey): primera aproximación, contextualizar por segmento   ║",
    "║  · Z-score MAD: robusto (breakdown 50%), preferido al clásico       ║",
    "║  · Chauvenet / Grubbs: solo bajo normalidad aproximada              ║",
    "║                                                                      ║",
    "║  【 DETECCIÓN MULTIVARIADA 】                                        ║",
    "║  · Mahalanobis MCD: siempre preferir sobre la clásica               ║",
    "║  · LOF: cuando hay densidades heterogéneas                          ║",
    "║  · Isolation Forest: escalable, no paramétrico                      ║",
    "║  · DBSCAN, One-Class SVM: alternativas no paramétricas              ║",
    "║  · CONSENSO: usar múltiples métodos y requerir ≥ 2 votos            ║",
    "║                                                                      ║",
    "║  【 CLASIFICAR ANTES DE TRATAR 】                                    ║",
    "║  · ¿Físicamente posible?      NO → corregir / marcar NA             ║",
    "║  · ¿Económicamente plausible? NO → investigar                       ║",
    "║  · ¿Es el objeto del análisis (fraude/default)? SÍ → PRESERVAR      ║",
    "║  · ¿El modelo es robusto? SÍ → mantener datos originales            ║",
    "║                                                                      ║",
    "║  【 TRATAMIENTO 】                                                   ║",
    "║  · Trimming: solo para errores verificados                          ║",
    "║  · Winsorización: preserva n, limita influencia                     ║",
    "║  · log / Box-Cox / Yeo-Johnson: normalizan y reducen curtosis       ║",
    "║  · Flag binario: preserva información del estatus de outlier        ║",
    "║  · Modelos robustos (Huber, MM, LTS): evita el dilema eliminar     ║",
    "║                                                                      ║",
    "║  【 AUDITABILIDAD 】                                                 ║",
    "║  · Documentar TODA transformación en log de auditoría               ║",
    "║  · Registrar: valor original, regla, decisión, justificación        ║",
    "║  · Exigencia regulatoria: BCRA, Basilea III-IV, IFRS 9              ║",
    "║                                                                      ║",
    "╚══════════════════════════════════════════════════════════════════════╝",
]
print("\n".join(sintesis))

---

### Lecturas recomendadas

- **Aggarwal, C. C.** (2017). *Outlier Analysis* (2nd ed.). Springer.
- **Rousseeuw, P. J. & Leroy, A. M.** (1987). *Robust Regression and Outlier Detection*. Wiley.
- **Huber, P. J. & Ronchetti, E. M.** (2009). *Robust Statistics* (2nd ed.). Wiley.
- **Nigrini, M. J.** (2012). *Benford's Law: Applications for Forensic Accounting, Auditing, and Fraud Detection*. Wiley.
- **Chandola, V., Banerjee, A. & Kumar, V.** (2009). Anomaly Detection: A Survey. *ACM Computing Surveys*, 41(3).

### Bibliotecas utilizadas

| Biblioteca | Uso principal |
|------------|---------------|
| `scipy.stats` | Tests (Shapiro, Jarque-Bera, Grubbs, Benford), Box-Cox |
| `sklearn.covariance.MinCovDet` | Estimador MCD (FAST-MCD de Rousseeuw-Van Driessen) |
| `sklearn.ensemble.IsolationForest` | Detección basada en aislamiento |
| `sklearn.neighbors.LocalOutlierFactor` | LOF de Breunig et al. |
| `sklearn.svm.OneClassSVM` | Detección por frontera no paramétrica |
| `sklearn.cluster.DBSCAN` | Clustering con detección implícita de ruido |
| `sklearn.preprocessing.PowerTransformer` | Yeo-Johnson y Box-Cox |
| `statsmodels.robust.RLM` | Regresión M de Huber y Tukey biweight |

---
> **Dr. Darío Ezequiel Díaz** · Maestría en Minería de Datos · UTN FRRo · 2026